In [ ]:
from rag_system import RAGSystem

# Initialize RAG system (load PDFs and build FAISS)
rag = RAGSystem(folder_path="documents")

Loading PDFs and creating chunks...
Processing: Aniket_CV.pdf
Total chunks created: 6
Building FAISS index...
RAG system ready.


In [8]:
# Query
reply = rag.answer_query("Is Aniket into swimming?")
print(reply)

 Yes, based on the information provided in [Document: Aniket_CV.pdf - Page: 2], Aniket has participated and won medals in swimming events. Specifically, he secured double victories by winning two Gold medals in the 200m Medley Relay and 400m Relay. However, there is no direct mention of his current involvement or status in swimming.




In [1]:
!pip install gradio

  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
   ---------------------------------------- 0.0/60.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/60.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/60.2 MB 2.8 MB/s eta 0:00:22
    --------------------------------------- 1.3/60.2 MB 2.6 MB/s eta 0:00:23
   - -------------------------------------- 1.8/60.2 MB 2.6 MB/s eta 0:00:23
   - -------------------------------------- 1.8/60.2 MB 2.6 MB/s eta 0:00:23
   - -------------------------------------- 2.4/60.2 MB 2.1 MB/s eta 0:00:28
   - -------------------------------------- 2.9/60.2 MB 2.2 MB/s eta 0:00:27
   -- ------------------------------------- 3.4/60.2 MB 2.2 MB/s eta 0:00:26
   -- ------------------------------------- 3.9/60.2 MB 2.2 MB/s eta 0:00:26
   -- ------------------------------------- 4.5/60.2 MB 2.3 MB/s eta 0:00:25
   --- ------------------------------------ 5.0/60.2 MB 2.3 MB/s eta 0:00:25
   --- -----

In [ ]:
# web_chatbot_gradio_ready.py
import gradio as gr
from rag_system import RAGSystem

# -------------------------------
# Initialize RAG system
# -------------------------------
rag = RAGSystem(folder_path="documents")  # Change folder path if needed

# -------------------------------
# Chat history storage
# -------------------------------
# Gradio handles session state for multiple users automatically
def respond(user_input, chat_history):
    if user_input.lower() in ["exit", "quit"]:
        chat_history.append({"role": "system", "content": "Exiting chatbot. Refresh the page to start over."})
        return chat_history, ""

    # Combine last few turns for context (last 3 Q&A pairs)
    recent_history = "\n".join(
        [f"Q: {msg['content']}" if msg['role']=='user' else f"A: {msg['content']}" for msg in chat_history[-6:]]
    )
    context_query = recent_history + f"\nQ: {user_input}"

    # Get answer from RAG system
    answer = rag.answer_query(context_query, top_k=3)

    # Append new Q&A to chat history in message format
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": answer})

    return chat_history, ""

# -------------------------------
# Gradio interface
# -------------------------------
with gr.Blocks() as demo:
    gr.Markdown("# 📚 RAG PDF Chatbot")
    gr.Markdown("Ask questions about your documents. Answers include document names and page numbers.")
    
    chatbot = gr.Chatbot(label="Chat History", type="messages")
    msg = gr.Textbox(label="Your question", placeholder="Type your question here...")
    clear = gr.Button("Clear Chat")

    # Submit user input
    msg.submit(respond, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: [], None, chatbot)

# -------------------------------
# Launch web server
# -------------------------------
# server_name="0.0.0.0" allows access from outside the host machine
demo.launch(server_name="0.0.0.0", server_port=7860)


Loading PDFs and creating chunks...
Processing: Aniket_CV.pdf
Total chunks: 6
Building or loading FAISS index...
Building FAISS index from scratch...
RAG system ready.


C:\Users\anike\AppData\Local\Temp\ipykernel_10140\2867017514.py:36: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History")


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


: 